# Public Debt Analysis: PostgreSQL + Python (Pandas) EDA

**Objective:** Examine the long-run trend, growth, and composition (domestic vs external) of the public debt stock.

**Workflow**
1. Data collection (official public debt statistics, supplied as CSV)
2. Database development (PostgreSQL table with integrity constraints)
3. Data import and validation (SQL checks)
4. Data extraction (Python -> PostgreSQL)
5. Data preparation (Pandas)
6. Exploratory data analysis
7. Visualisation (charts saved to `figures/`; processed data exported for Power BI)

> **Units:** debt values are used exactly as reported in the source file. They appear to be in millions of local currency (KES) - **confirm against the source's metadata** and edit `UNIT_LABEL` below.

## 1. Setup and configuration

In [ ]:
import os
import sqlite3
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid", context="notebook")

# ------------------------- CONFIG -------------------------
DATA_PATH  = Path("data/public_debt.csv")
FIG_DIR    = Path("figures");  FIG_DIR.mkdir(exist_ok=True)
OUT_DIR    = Path("outputs");  OUT_DIR.mkdir(exist_ok=True)
UNIT_LABEL = "KES million"          # <-- confirm units from the source

# True  -> use PostgreSQL (credentials from environment variables)
# False -> use a local SQLite file so the notebook runs anywhere (e.g. a GitHub clone without Postgres)
USE_POSTGRES = os.getenv("USE_POSTGRES", "true").lower() == "true"

PG_CONFIG = dict(
    host     = os.getenv("PGHOST", "localhost"),
    port     = int(os.getenv("PGPORT", 5432)),
    dbname   = os.getenv("PGDATABASE", "public_debt_db"),
    user     = os.getenv("PGUSER", "postgres"),
    password = os.getenv("PGPASSWORD", ""),     # never hard-code passwords in a public repo
)
RECON_TOLERANCE = 1.0   # allowed rounding difference (domestic + external vs total)

## 2. Data collection
Load the raw CSV and standardise the column names (the source headers contain stray spaces).

In [ ]:
raw = pd.read_csv(DATA_PATH)
raw.columns = (raw.columns.str.strip().str.lower())
raw = raw.rename(columns={
    "year": "report_year", "month": "report_month",
    "domestic_debt": "domestic_debt", "external_debt": "external_debt", "total": "total_debt",
})
print("Shape:", raw.shape)
raw.head()

In [ ]:
raw.info()
raw.isna().sum()

## 3. Database development
Create the PostgreSQL database once (run in `psql` or pgAdmin), then the notebook creates the table:

```sql
CREATE DATABASE public_debt_db;
```

The table enforces the key rules **at database level**: month 1-12, non-negative values, and one record per reporting period (`UNIQUE (year, month)`). The same DDL is stored in `sql/schema.sql`.

In [ ]:
DDL_POSTGRES = """
CREATE TABLE IF NOT EXISTS public_debt (
    id             SERIAL PRIMARY KEY,
    report_year    INTEGER       NOT NULL CHECK (report_year BETWEEN 1990 AND 2100),
    report_month   INTEGER       NOT NULL CHECK (report_month BETWEEN 1 AND 12),
    domestic_debt  NUMERIC(18,2) NOT NULL CHECK (domestic_debt >= 0),
    external_debt  NUMERIC(18,2) NOT NULL CHECK (external_debt >= 0),
    total_debt     NUMERIC(18,2) NOT NULL CHECK (total_debt >= 0),
    CONSTRAINT uq_public_debt_period UNIQUE (report_year, report_month)
);
"""
DDL_SQLITE = DDL_POSTGRES.replace("SERIAL PRIMARY KEY", "INTEGER PRIMARY KEY AUTOINCREMENT")

def get_connection():
    """Return (connection, placeholder). Placeholder differs by driver (%s vs ?)."""
    if USE_POSTGRES:
        import psycopg2
        return psycopg2.connect(**PG_CONFIG), "%s"
    return sqlite3.connect(OUT_DIR / "public_debt.sqlite"), "?"

conn, PH = get_connection()
print("Backend:", "PostgreSQL" if USE_POSTGRES else "SQLite (fallback)")

cur = conn.cursor()
cur.execute(DDL_POSTGRES if USE_POSTGRES else DDL_SQLITE)
conn.commit()

## 4. Data import
The load is **idempotent**: the table is emptied first, so re-running the notebook never creates duplicates.

In [ ]:
cur.execute("DELETE FROM public_debt")

insert_sql = f"""
INSERT INTO public_debt (report_year, report_month, domestic_debt, external_debt, total_debt)
VALUES ({PH}, {PH}, {PH}, {PH}, {PH})
"""
records = [
    (int(r.report_year), int(r.report_month),
     float(r.domestic_debt), float(r.external_debt), float(r.total_debt))
    for r in raw.itertuples(index=False)
]
cur.executemany(insert_sql, records)
conn.commit()
print(f"Inserted {len(records)} rows")

## 5. Data validation (SQL checks)
Each check returns a PASS / FAIL result. Constraints on the table would already block most bad rows on insert; these checks document and confirm data quality explicitly.

In [ ]:
def q(sql):
    return pd.read_sql_query(sql, conn)

checks = {}

# 5.1 Record count verification (CSV vs database)
db_count = q("SELECT COUNT(*) AS n FROM public_debt").iloc[0, 0]
checks["Record count (CSV = DB)"] = (db_count == len(raw), f"CSV={len(raw)}, DB={db_count}")

# 5.2 Duplicate reporting periods
dups = q("""
    SELECT report_year, report_month, COUNT(*) AS n
    FROM public_debt GROUP BY report_year, report_month HAVING COUNT(*) > 1
""")
checks["No duplicate year-month"] = (dups.empty, f"{len(dups)} duplicate periods")

# 5.3 Month-range validation (1-12)
bad_months = q("SELECT * FROM public_debt WHERE report_month NOT BETWEEN 1 AND 12")
checks["Months within 1-12"] = (bad_months.empty, f"{len(bad_months)} invalid rows")

# 5.4 Non-negative debt values
neg = q("""
    SELECT * FROM public_debt
    WHERE domestic_debt < 0 OR external_debt < 0 OR total_debt < 0
""")
checks["Debt values non-negative"] = (neg.empty, f"{len(neg)} negative rows")

# 5.5 Reconciliation: domestic + external = total (within rounding tolerance)
recon = q(f"""
    SELECT report_year, report_month, domestic_debt, external_debt, total_debt,
           (domestic_debt + external_debt - total_debt) AS difference
    FROM public_debt
    WHERE ABS(domestic_debt + external_debt - total_debt) > {RECON_TOLERANCE}
""")
max_diff = q("SELECT MAX(ABS(domestic_debt + external_debt - total_debt)) AS d FROM public_debt").iloc[0, 0]
checks["Domestic + External = Total"] = (recon.empty, f"{len(recon)} breaks; max abs diff = {float(max_diff):,.4f}")

validation = pd.DataFrame(
    [(k, "PASS" if ok else "FAIL", msg) for k, (ok, msg) in checks.items()],
    columns=["Check", "Result", "Detail"],
)
validation

In [ ]:
# 5.6 Additional check: coverage of the reporting calendar (gaps between first and last period)
cov = q("SELECT report_year, report_month FROM public_debt ORDER BY report_year, report_month")
cov["date"] = pd.to_datetime(dict(year=cov.report_year, month=cov.report_month, day=1))
full = pd.date_range(cov.date.min(), cov.date.max(), freq="MS")
missing = full.difference(cov.date)
print(f"Period covered : {cov.date.min():%b %Y} to {cov.date.max():%b %Y}")
print(f"Months present : {len(cov)} of {len(full)} possible ({len(missing)} missing)")
print("Missing months by year:")
print(pd.Series(missing.year).value_counts().sort_index().to_string())

**Note on coverage:** reporting is irregular in 1999-2000 (only a few observations), and is continuous monthly from 2001. Growth rates that need a 12-month lag are therefore only calculated where the prior-year month exists.

## 6. Data extraction (PostgreSQL -> Pandas)

In [ ]:
df = q("""
    SELECT report_year  AS year,
           report_month AS month,
           domestic_debt, external_debt, total_debt
    FROM public_debt
    ORDER BY report_year, report_month
""")
for c in ["domestic_debt", "external_debt", "total_debt"]:
    df[c] = df[c].astype(float)      # NUMERIC arrives as Decimal from psycopg2
print(df.shape)
df.head()

## 7. Data preparation
- Build a reporting date from year and month
- Domestic and external debt shares
- Month-on-month and year-on-year changes (calendar-aware, so gaps do not create false growth rates)
- Annual (year-end) series

In [ ]:
df["date"] = pd.to_datetime(dict(year=df.year, month=df.month, day=1)) + pd.offsets.MonthEnd(0)
df = df.sort_values("date").set_index("date")

# Debt shares (%)
df["domestic_share"] = df.domestic_debt / df.total_debt * 100
df["external_share"] = df.external_debt / df.total_debt * 100

# Calendar-aware changes: reindex to a full month-end calendar so shift(1)/shift(12) mean 1 / 12 real months
cal = df[["domestic_debt", "external_debt", "total_debt"]].reindex(
    pd.date_range(df.index.min(), df.index.max(), freq="ME"))
for c in ["domestic_debt", "external_debt", "total_debt"]:
    df[f"{c}_mom_pct"] = (cal[c].pct_change(1,  fill_method=None) * 100).reindex(df.index)
    df[f"{c}_yoy_pct"] = (cal[c].pct_change(12, fill_method=None) * 100).reindex(df.index)
    df[f"{c}_yoy_abs"] = (cal[c].diff(12)).reindex(df.index)

df["ext_to_dom_ratio"] = df.external_debt / df.domestic_debt
df.tail()

In [ ]:
# Annual series: last available observation in each year (December for full years; latest month for 1999/2000/2026)
annual = (df.reset_index()
            .sort_values("date")
            .groupby("year")
            .tail(1)
            .set_index("year")
            [["date", "domestic_debt", "external_debt", "total_debt", "domestic_share", "external_share"]])
annual["last_month"] = annual.date.dt.month
annual["is_full_year"] = annual.last_month == 12
for c in ["domestic_debt", "external_debt", "total_debt"]:
    annual[f"{c}_growth_pct"] = annual[c].pct_change() * 100
    annual[f"{c}_change"]     = annual[c].diff()
annual.tail(10)

## 8. Exploratory data analysis

### 8.1 Summary statistics

In [ ]:
df[["domestic_debt", "external_debt", "total_debt", "domestic_share", "external_share"]].describe().T

### 8.2 Historical milestones

In [ ]:
first, last = df.iloc[0], df.iloc[-1]
peak_total = df.total_debt.idxmax()

print(f"First observation : {df.index[0]:%b %Y}  total = {first.total_debt:,.0f} {UNIT_LABEL}")
print(f"Latest observation: {df.index[-1]:%b %Y}  total = {last.total_debt:,.0f} {UNIT_LABEL}")
print(f"Peak total debt   : {peak_total:%b %Y} ({df.total_debt.max():,.0f})")
print(f"Growth since start: {last.total_debt / first.total_debt:,.1f}x "
      f"({(last.total_debt / first.total_debt - 1) * 100:,.0f}%)")

years = (df.index[-1] - df.index[0]).days / 365.25
cagr = lambda a, b: ((b / a) ** (1 / years) - 1) * 100
print(f"\nCompound annual growth rate ({df.index[0]:%b %Y}-{df.index[-1]:%b %Y}):")
print(f"  Total    : {cagr(first.total_debt,    last.total_debt):.2f}%")
print(f"  Domestic : {cagr(first.domestic_debt, last.domestic_debt):.2f}%")
print(f"  External : {cagr(first.external_debt, last.external_debt):.2f}%")

# Round-number milestones
for m in [1e6, 2.5e6, 5e6, 10e6]:
    hit = df[df.total_debt >= m]
    if not hit.empty:
        print(f"Total debt first exceeded {m:>12,.0f}: {hit.index[0]:%b %Y}")

### 8.3 Growth of the debt stock

In [ ]:
display_cols = ["total_debt", "total_debt_growth_pct", "domestic_debt_growth_pct", "external_debt_growth_pct"]
print("Annual growth (year-end vs prior year-end; 2026 is partial-year):")
annual[display_cols].round(2).tail(15)

In [ ]:
full_years = annual[annual.is_full_year & annual.total_debt_growth_pct.notna()]
print("Average annual growth of total debt, full years:")
print(f"  Overall                : {full_years.total_debt_growth_pct.mean():.2f}%")
for lo, hi in [(2001, 2010), (2011, 2015), (2016, 2020), (2021, 2025)]:
    s = full_years.loc[lo:hi, "total_debt_growth_pct"]
    print(f"  {lo}-{hi}              : {s.mean():.2f}%")

print("\nFastest growth years (total debt):")
print(full_years.nlargest(5, "total_debt_growth_pct")[["total_debt_growth_pct", "total_debt_change"]].round(2))
print("\nLargest absolute annual additions:")
print(full_years.nlargest(5, "total_debt_change")[["total_debt_change", "total_debt_growth_pct"]].round(2))

In [ ]:
# Largest single-month movements (only where the previous month is available)
mom = df["total_debt_mom_pct"].dropna()
print("Biggest monthly increases (%):")
print(mom.nlargest(5).round(2))
print("\nBiggest monthly decreases (%):")
print(mom.nsmallest(5).round(2))

### 8.4 Changes in debt composition (domestic vs external)

In [ ]:
comp = annual[["domestic_share", "external_share"]].round(1)
comp["dominant"] = np.where(comp.domestic_share > comp.external_share, "Domestic", "External")
comp.tail(15)

In [ ]:
# Month-to-month the lead flips often when the two are close, so judge dominance on YEAR-END data
ye_dom = pd.Series(np.where(annual.domestic_debt > annual.external_debt, "Domestic", "External"), index=annual.index)
changes = ye_dom[ye_dom != ye_dom.shift()]
print("Larger component at each year-end (changes only):")
for yr, v in changes.items():
    print(f"  {yr}: {v} debt is the larger component")
months_dom = (df.domestic_debt > df.external_debt).mean() * 100
print(f"\nDomestic exceeded external in {months_dom:.0f}% of all reported months")

print(f"\nExternal share: min {df.external_share.min():.1f}% ({df.external_share.idxmin():%b %Y}), "
      f"max {df.external_share.max():.1f}% ({df.external_share.idxmax():%b %Y}), "
      f"latest {df.external_share.iloc[-1]:.1f}%")
print(f"Domestic share: min {df.domestic_share.min():.1f}% ({df.domestic_share.idxmin():%b %Y}), "
      f"max {df.domestic_share.max():.1f}% ({df.domestic_share.idxmax():%b %Y}), "
      f"latest {df.domestic_share.iloc[-1]:.1f}%")

In [ ]:
# Contribution to growth: how much of the total increase came from each component, by period
def contribution(start, end):
    a, b = df.loc[:start].iloc[-1], df.loc[:end].iloc[-1]
    inc = b.total_debt - a.total_debt
    return pd.Series({
        "total_increase": inc,
        "domestic_contrib_%": (b.domestic_debt - a.domestic_debt) / inc * 100,
        "external_contrib_%": (b.external_debt - a.external_debt) / inc * 100,
    })

periods = {"2001-2010": ("2000-12-31", "2010-12-31"), "2011-2015": ("2010-12-31", "2015-12-31"),
           "2016-2020": ("2015-12-31", "2020-12-31"), "2021-latest": ("2020-12-31", df.index[-1])}
pd.DataFrame({k: contribution(*v) for k, v in periods.items()}).T.round(1)

### 8.5 Correlation of growth rates and ratio trend

In [ ]:
yoy = df[["domestic_debt_yoy_pct", "external_debt_yoy_pct", "total_debt_yoy_pct"]].dropna()
print("Correlation of YoY growth rates:")
yoy.corr().round(2)

## 9. Visualisation

In [ ]:
def fmt_millions(x, pos):
    # axis shown in trillions if values are in millions
    return f"{x/1e6:,.0f}"

AXIS_LABEL = "Debt (trillions)" if "million" in UNIT_LABEL.lower() else f"Debt ({UNIT_LABEL})"
COLORS = {"domestic": "#1f77b4", "external": "#d95f02", "total": "#2b2b2b"}

def finish(fig, name):
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

### 9.1 Total public debt over time

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(df.index, df.total_debt, color=COLORS["total"], lw=2.2)
ax.fill_between(df.index, df.total_debt, alpha=0.1, color=COLORS["total"])
ax.yaxis.set_major_formatter(mtick.FuncFormatter(fmt_millions))
ax.set(title="Total Public Debt Over Time", xlabel="", ylabel=AXIS_LABEL)
ax.annotate(f"{df.total_debt.iloc[-1]/1e6:,.2f}T\n({df.index[-1]:%b %Y})",
            xy=(df.index[-1], df.total_debt.iloc[-1]), xytext=(-70, -10),
            textcoords="offset points", ha="center", fontsize=10, fontweight="bold")
finish(fig, "01_total_debt_over_time")

### 9.2 Domestic versus external debt

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(df.index, df.domestic_debt, label="Domestic debt", color=COLORS["domestic"], lw=2)
ax.plot(df.index, df.external_debt, label="External debt", color=COLORS["external"], lw=2)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(fmt_millions))
ax.set(title="Domestic vs External Public Debt", ylabel=AXIS_LABEL)
ax.legend(frameon=False)
finish(fig, "02_domestic_vs_external")

### 9.3 Debt composition over time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].stackplot(df.index, df.domestic_share, df.external_share,
                  labels=["Domestic", "External"], colors=[COLORS["domestic"], COLORS["external"]], alpha=0.85)
axes[0].axhline(50, color="white", ls="--", lw=1)
axes[0].set(title="Debt Composition (% of total)", ylim=(0, 100), ylabel="Share of total debt (%)")
axes[0].legend(loc="lower left", frameon=False)

ann = annual[annual.is_full_year]
axes[1].bar(ann.index, ann.domestic_debt, label="Domestic", color=COLORS["domestic"])
axes[1].bar(ann.index, ann.external_debt, bottom=ann.domestic_debt, label="External", color=COLORS["external"])
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(fmt_millions))
axes[1].set(title="Year-end Debt Stock by Component", ylabel=AXIS_LABEL)
axes[1].legend(frameon=False)
finish(fig, "03_debt_composition")

### 9.4 Annual debt growth

In [ ]:
g = annual[annual.is_full_year & annual.total_debt_growth_pct.notna()]
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(g.index, g.total_debt_growth_pct, color=COLORS["total"], alpha=0.85, label="Total")
ax.plot(g.index, g.domestic_debt_growth_pct, marker="o", color=COLORS["domestic"], label="Domestic")
ax.plot(g.index, g.external_debt_growth_pct, marker="o", color=COLORS["external"], label="External")
ax.axhline(0, color="black", lw=0.8)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set(title="Annual Growth of Public Debt (year-end to year-end)", ylabel="Growth (%)")
ax.legend(frameon=False, ncol=3)
finish(fig, "04_annual_debt_growth")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(g.index, g.total_debt_change, color=np.where(g.total_debt_change >= 0, COLORS["total"], "#c0392b"))
ax.yaxis.set_major_formatter(mtick.FuncFormatter(fmt_millions))
ax.set(title="Annual Increase in Total Debt Stock", ylabel=f"Change ({'trillions' if 'million' in UNIT_LABEL.lower() else UNIT_LABEL})")
finish(fig, "05_annual_debt_change")

### 9.5 Year-on-year growth (monthly view)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(df.index, df.domestic_debt_yoy_pct, label="Domestic", color=COLORS["domestic"])
ax.plot(df.index, df.external_debt_yoy_pct, label="External", color=COLORS["external"])
ax.plot(df.index, df.total_debt_yoy_pct, label="Total", color=COLORS["total"], lw=2.2)
ax.axhline(0, color="black", lw=0.8)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set(title="Year-on-Year Growth in Public Debt", ylabel="YoY change (%)")
ax.legend(frameon=False, ncol=3)
finish(fig, "06_yoy_growth")

### 9.6 Debt-to-GDP (optional - requires GDP data)

The supplied dataset has no GDP series. To enable this section, add `data/gdp.csv` with columns `Year,GDP` (**same unit as debt**, nominal, annual). Without the file the cell is skipped.

In [ ]:
GDP_PATH = Path("data/gdp.csv")
if GDP_PATH.exists():
    gdp = pd.read_csv(GDP_PATH)
    gdp.columns = gdp.columns.str.strip().str.lower()
    ye = annual[annual.is_full_year][["total_debt", "domestic_debt", "external_debt"]].join(
        gdp.set_index("year")["gdp"], how="inner")
    ye["debt_to_gdp"]          = ye.total_debt    / ye.gdp * 100
    ye["domestic_to_gdp"]      = ye.domestic_debt / ye.gdp * 100
    ye["external_to_gdp"]      = ye.external_debt / ye.gdp * 100

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(ye.index, ye.debt_to_gdp, marker="o", color=COLORS["total"], label="Total")
    ax.plot(ye.index, ye.domestic_to_gdp, color=COLORS["domestic"], label="Domestic")
    ax.plot(ye.index, ye.external_to_gdp, color=COLORS["external"], label="External")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set(title="Public Debt-to-GDP", ylabel="% of GDP")
    ax.legend(frameon=False)
    finish(fig, "07_debt_to_gdp")
else:
    print("data/gdp.csv not found - Debt-to-GDP section skipped.")

## 10. Key findings (generated from the data)

In [ ]:
latest = df.iloc[-1]
ann_full = annual[annual.is_full_year & annual.total_debt_growth_pct.notna()]
latest_yoy = df.total_debt_yoy_pct.dropna().iloc[-1]
sw = changes.iloc[1:]   # skip the first entry (starting position, not a change)

findings = [
    f"Total public debt rose from {first.total_debt:,.0f} ({df.index[0]:%b %Y}) to {latest.total_debt:,.0f} "
    f"({df.index[-1]:%b %Y}) {UNIT_LABEL} - about {latest.total_debt/first.total_debt:,.1f}x.",
    f"Long-run compound growth: total {cagr(first.total_debt, latest.total_debt):.1f}% a year "
    f"(domestic {cagr(first.domestic_debt, latest.domestic_debt):.1f}%, external {cagr(first.external_debt, latest.external_debt):.1f}%).",
    f"Average annual growth in full years: {ann_full.total_debt_growth_pct.mean():.1f}%; "
    f"fastest year {ann_full.total_debt_growth_pct.idxmax()} ({ann_full.total_debt_growth_pct.max():.1f}%).",
    f"Latest year-on-year growth in total debt: {latest_yoy:.1f}%.",
    f"Current composition: {latest.domestic_share:.1f}% domestic / {latest.external_share:.1f}% external.",
]
for yr, v in sw.items():
    findings.append(f"Composition shift: {v} debt became the larger component at year-end {yr}.")
for f in findings:
    print("-", f)

## 11. Export processed data (for Power BI)

In [ ]:
export = df.reset_index().rename(columns={"index": "date"})
export.to_csv(OUT_DIR / "public_debt_processed.csv", index=False)
annual.reset_index().to_csv(OUT_DIR / "public_debt_annual.csv", index=False)
print("Saved:", *[p.name for p in OUT_DIR.glob("*.csv")])

cur.close(); conn.close()

## 12. Next steps
- Add GDP data (`data/gdp.csv`) to enable debt-to-GDP analysis
- Build the Power BI dashboard on `outputs/public_debt_processed.csv` (or connect Power BI directly to PostgreSQL) with KPI cards, year slicer, and domestic/external filter
- Add debt service, interest cost, and currency composition if available